In [2]:
import pandas as pd

In [3]:
# Load files
df = pd.read_csv("GDSC_DATASET.csv")
df2 = pd.read_csv("genomic_features.csv")
df3 = pd.read_csv("model_list_latest.csv")
df4 = pd.read_csv("Model.csv")

C:\Users\shibu\AppData\Local\Temp\ipykernel_13808\4134805013.py:3: DtypeWarning: Columns (0: Recurrent Gain Loss, 1: Genes in Segment) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv("genomic_features.csv")


In [4]:
# Basic shape check
print("df shape:", df.shape)
print("df2 shape:", df2.shape)
print("df3 shape:", df3.shape)
print("df4 shape:", df4.shape)

# Check whether COSMIC_ID exists
for name, temp_df in {
    "df": df,
    "df2": df2,
    "df3": df3,
    "df4": df4
}.items():
    print(f"\n{name} has COSMIC_ID:", "COSMIC_ID" in temp_df.columns)

df shape: (242035, 19)
df2 shape: (698000, 9)
df3 shape: (2266, 96)
df4 shape: (2132, 49)

df has COSMIC_ID: True

df2 has COSMIC_ID: False

df3 has COSMIC_ID: True

df4 has COSMIC_ID: False


Checked the df2 and df4, it has cosmic id but instead of being COSMIC_ID, It is COSMICID

In [5]:
# Standardize COSMIC key column names
df2 = df2.rename(columns={"COSMIC ID": "COSMIC_ID"})
df4 = df4.rename(columns={"COSMICID": "COSMIC_ID"})

'''
# Check again that all files now have COSMIC_ID
for name, temp_df in {
    "df": df,
    "df2": df2,
    "df3": df3,
    "df4": df4
}.items():
    print(f"{name} has COSMIC_ID:", "COSMIC_ID" in temp_df.columns)'''

# Standardize COSMIC_ID type across all files
for temp_df in [df, df2, df3, df4]:
    temp_df["COSMIC_ID"] = pd.to_numeric(temp_df["COSMIC_ID"], errors="coerce")

# Check missing COSMIC_ID values
for name, temp_df in {
    "df": df,
    "df2": df2,
    "df3": df3,
    "df4": df4
}.items():
    print(f"{name} missing COSMIC_ID:", temp_df["COSMIC_ID"].isna().sum())

print("====Sepator====")

# Check unique COSMIC_ID counts
for name, temp_df in {
    "df": df,
    "df2": df2,
    "df3": df3,
    "df4": df4
}.items():
    print(f"{name} unique COSMIC_ID:", temp_df["COSMIC_ID"].nunique())

print("====Sepator====")

# Check duplicate COSMIC_ID rows
for name, temp_df in {
    "df": df,
    "df2": df2,
    "df3": df3,
    "df4": df4
}.items():
    dup_count = temp_df["COSMIC_ID"].duplicated().sum()
    print(f"{name} duplicated COSMIC_ID rows:", dup_count)

# Overlap with main dataset
df_ids  = set(df["COSMIC_ID"].dropna().unique())
df2_ids = set(df2["COSMIC_ID"].dropna().unique())
df3_ids = set(df3["COSMIC_ID"].dropna().unique())
df4_ids = set(df4["COSMIC_ID"].dropna().unique())

print("====Sepator====")

print("\nOverlap df vs df2:", len(df_ids & df2_ids))
print("Overlap df vs df3:", len(df_ids & df3_ids))
print("Overlap df vs df4:", len(df_ids & df4_ids))

print("\ndf IDs not in df2:", len(df_ids - df2_ids))
print("df IDs not in df3:", len(df_ids - df3_ids))
print("df IDs not in df4:", len(df_ids - df4_ids))

df missing COSMIC_ID: 0
df2 missing COSMIC_ID: 0
df3 missing COSMIC_ID: 1144
df4 missing COSMIC_ID: 1155
====Sepator====
df unique COSMIC_ID: 969
df2 unique COSMIC_ID: 969
df3 unique COSMIC_ID: 1122
df4 unique COSMIC_ID: 977
====Sepator====
df duplicated COSMIC_ID rows: 241066
df2 duplicated COSMIC_ID rows: 697031
df3 duplicated COSMIC_ID rows: 1143
df4 duplicated COSMIC_ID rows: 1154
====Sepator====

Overlap df vs df2: 969
Overlap df vs df3: 969
Overlap df vs df4: 946

df IDs not in df2: 0
df IDs not in df3: 0
df IDs not in df4: 23


# Validation 

We have to take care of the duplicate cosmic_id on the supplemental csv because if we dont take care of it, it would affect inflate the data unnceassarly.
FOr example:
-  In df- COSMIC_ID 1234 appears 100 times because that cell line was tested with many drugs. This is okay because we need that 
- But if in df2- COSMIC_ID 1234 appears 3 times. Then after merging, those 100 rows can turn into 300 rows if pandas matches all combinations. This is bad

In [6]:
# --- 1) check duplicate COSMIC_ID only among non-missing rows ---
for name, temp_df in {"df": df, "df2": df2, "df3": df3, "df4": df4}.items():
    temp = temp_df[temp_df["COSMIC_ID"].notna()].copy()
    dup_rows = temp[temp["COSMIC_ID"].duplicated(keep=False)]

    print(f"\n{name}")
    print("non-missing COSMIC_ID rows:", temp.shape[0])
    print("unique non-missing COSMIC_ID:", temp["COSMIC_ID"].nunique())
    print("duplicated non-missing COSMIC_ID rows:", dup_rows.shape[0])
    print("duplicated non-missing COSMIC_ID unique IDs:", dup_rows["COSMIC_ID"].nunique())

# --- 2) overlap check using only non-missing COSMIC_ID ---
df_ids  = set(df.loc[df["COSMIC_ID"].notna(), "COSMIC_ID"].unique())
df2_ids = set(df2.loc[df2["COSMIC_ID"].notna(), "COSMIC_ID"].unique())
df3_ids = set(df3.loc[df3["COSMIC_ID"].notna(), "COSMIC_ID"].unique())
df4_ids = set(df4.loc[df4["COSMIC_ID"].notna(), "COSMIC_ID"].unique())

print("\nOverlap df vs df2:", len(df_ids & df2_ids))
print("Overlap df vs df3:", len(df_ids & df3_ids))
print("Overlap df vs df4:", len(df_ids & df4_ids))

print("\ndf IDs not in df2:", len(df_ids - df2_ids))
print("df IDs not in df3:", len(df_ids - df3_ids))
print("df IDs not in df4:", len(df_ids - df4_ids))

# --- 3) inspect whether df3 and df4 are one-row-per-cell-line after removing missing IDs ---
print("\ndf3 rows after dropping missing COSMIC_ID:", df3[df3['COSMIC_ID'].notna()].shape)
print("df4 rows after dropping missing COSMIC_ID:", df4[df4['COSMIC_ID'].notna()].shape)

# --- 4) quick preview of repeated real COSMIC_IDs if any exist in df3 / df4 ---
for name, temp_df in {"df3": df3, "df4": df4}.items():
    temp = temp_df[temp_df["COSMIC_ID"].notna()].copy()
    dup_rows = temp[temp["COSMIC_ID"].duplicated(keep=False)].sort_values("COSMIC_ID")
    print(f"\n===== {name} repeated real COSMIC_ID examples =====")
    print(dup_rows.head(10))


df
non-missing COSMIC_ID rows: 242035
unique non-missing COSMIC_ID: 969
duplicated non-missing COSMIC_ID rows: 242035
duplicated non-missing COSMIC_ID unique IDs: 969

df2
non-missing COSMIC_ID rows: 698000
unique non-missing COSMIC_ID: 969
duplicated non-missing COSMIC_ID rows: 698000
duplicated non-missing COSMIC_ID unique IDs: 969

df3
non-missing COSMIC_ID rows: 1122
unique non-missing COSMIC_ID: 1122
duplicated non-missing COSMIC_ID rows: 0
duplicated non-missing COSMIC_ID unique IDs: 0

df4
non-missing COSMIC_ID rows: 977
unique non-missing COSMIC_ID: 977
duplicated non-missing COSMIC_ID rows: 0
duplicated non-missing COSMIC_ID unique IDs: 0

Overlap df vs df2: 969
Overlap df vs df3: 969
Overlap df vs df4: 946

df IDs not in df2: 0
df IDs not in df3: 0
df IDs not in df4: 23

df3 rows after dropping missing COSMIC_ID: (1122, 96)
df4 rows after dropping missing COSMIC_ID: (977, 49)

===== df3 repeated real COSMIC_ID examples =====
Empty DataFrame
Columns: [model_id, sample_id, pat

Inspect useful helper columns from df3 and df4
- This step is trying to figure out which columns from the helper datasets might actually help fill missing information in the main dataset df. The reason for doing this first is so we do not merge in a lot of extra columns blindly. It works by listing likely useful columns such as MSI, growth, tissue, or cancer-type related fields and making smaller helper tables with just those columns.

In [7]:
# --- 1) inspect useful columns from df3 and df4 that may help fill df ---
print("df columns:\n", df.columns.tolist(), "\n")

print("Possible helper columns from df3:")
print([
    "COSMIC_ID", "model_name", "tissue", "cancer_type", "cancer_type_detail",
    "growth_properties", "msi_status", "sample_site"
], "\n")

print("Possible helper columns from df4:")
print([
    "COSMIC_ID", "CellLineName", "StrippedCellLineName", "OncotreeLineage",
    "OncotreePrimaryDisease", "OncotreeSubtype", "GrowthPattern",
    "OnboardedMedia", "TissueOrigin"
], "\n")


df columns:
 ['COSMIC_ID', 'CELL_LINE_NAME', 'TCGA_DESC', 'DRUG_ID', 'DRUG_NAME', 'LN_IC50', 'AUC', 'Z_SCORE', 'GDSC Tissue descriptor 1', 'GDSC Tissue descriptor 2', 'Cancer Type (matching TCGA label)', 'Microsatellite instability Status (MSI)', 'Screen Medium', 'Growth Properties', 'CNA', 'Gene Expression', 'Methylation', 'TARGET', 'TARGET_PATHWAY'] 

Possible helper columns from df3:
['COSMIC_ID', 'model_name', 'tissue', 'cancer_type', 'cancer_type_detail', 'growth_properties', 'msi_status', 'sample_site'] 

Possible helper columns from df4:
['COSMIC_ID', 'CellLineName', 'StrippedCellLineName', 'OncotreeLineage', 'OncotreePrimaryDisease', 'OncotreeSubtype', 'GrowthPattern', 'OnboardedMedia', 'TissueOrigin'] 



Output explanation - The output tells us which columns are available in the supplementary datasets and whether they look relevant for filling missing values in df. This helps us decide which columns are worth bringing in and which ones are unrelated. In other words, the output is like a preview of possible sources for missing MSI, growth, tissue, or cancer-type information.

Test safe merges to confirm row count does not expand
- This step is checking whether merging the helper tables into df causes duplicate rows. The reason this matters is that if the row count increases, it usually means one COSMIC_ID is matching multiple rows in the helper file, which can corrupt the dataset. It works by doing a test merge and then comparing the number of rows before and after the merge.

In [8]:

# --- 2) create reduced helper tables with only likely useful columns ---
df3_small = df3[[
    "COSMIC_ID", "model_name", "tissue", "cancer_type",
    "cancer_type_detail", "growth_properties", "msi_status", "sample_site"
]].copy()

df4_small = df4[[
    "COSMIC_ID", "CellLineName", "StrippedCellLineName", "OncotreeLineage",
    "OncotreePrimaryDisease", "OncotreeSubtype", "GrowthPattern",
    "OnboardedMedia", "TissueOrigin"
]].copy()

# --- 3) safe merge test: row count should stay the same ---
print("Original df rows:", df.shape[0])

df_test3 = df.merge(df3_small, on="COSMIC_ID", how="left")
print("After merging df3_small:", df_test3.shape[0])

df_test4 = df.merge(df4_small, on="COSMIC_ID", how="left")
print("After merging df4_small:", df_test4.shape[0])

# --- 4) check how many rows got helper info after merge ---
print("\nFilled rows from df3 helper columns:")
print(df_test3[["model_name", "tissue", "cancer_type", "growth_properties", "msi_status"]].notna().sum())

print("\nFilled rows from df4 helper columns:")
print(df_test4[["CellLineName", "OncotreeLineage", "OncotreePrimaryDisease", "GrowthPattern", "OnboardedMedia"]].notna().sum())

# --- 5) compare a few rows to understand matching quality ---
print("\nSample df + df3 merged rows:")
print(df_test3[[
    "COSMIC_ID", "CELL_LINE_NAME", "TCGA_DESC",
    "Microsatellite instability Status (MSI)", "Growth Properties",
    "model_name", "cancer_type", "msi_status", "growth_properties"
]].head(10))

print("\nSample df + df4 merged rows:")
print(df_test4[[
    "COSMIC_ID", "CELL_LINE_NAME", "TCGA_DESC",
    "Screen Medium", "Growth Properties",
    "CellLineName", "OncotreePrimaryDisease", "GrowthPattern", "OnboardedMedia"
]].head(10))

Original df rows: 242035
After merging df3_small: 242035
After merging df4_small: 242035

Filled rows from df3 helper columns:
model_name           242035
tissue               242035
cancer_type          242035
growth_properties    242035
msi_status           240011
dtype: int64

Filled rows from df4 helper columns:
CellLineName              236302
OncotreeLineage           236302
OncotreePrimaryDisease    236302
GrowthPattern             236302
OnboardedMedia            178620
dtype: int64

Sample df + df3 merged rows:
   COSMIC_ID CELL_LINE_NAME     TCGA_DESC  \
0     683667         PFSK-1            MB   
1     684057            ES5  UNCLASSIFIED   
2     684059            ES7  UNCLASSIFIED   
3     684062          EW-11  UNCLASSIFIED   
4     684072        SK-ES-1  UNCLASSIFIED   
5     687448       COLO-829          SKCM   
6     687452           5637          BLCA   
7     687455            RT4          BLCA   
8     687457          SW780          BLCA   
9     687459         TCC

The output tells you whether the merge is structurally safe. If the row count stays the same before and after the merge, that usually means each row in df matched cleanly and the merge did not create duplicates. If the row count increases, that is a warning sign that some COSMIC_IDs are matching multiple rows in the helper table, which means we  should not trust that merge yet.

Test safe merges to confirm row count does not expand
- This step is checking whether merging the helper tables into df causes duplicate rows. The reason this matters is that if the row count increases, it usually means one COSMIC_ID is matching multiple rows in the helper file, which can corrupt the dataset. It works by doing a test merge and then comparing the number of rows before and after the merge.

In [9]:
# Missing values in df columns we may want to fill
fill_targets = [
    "TCGA_DESC",
    "GDSC Tissue descriptor 1",
    "GDSC Tissue descriptor 2",
    "Microsatellite instability Status (MSI)",
    "Screen Medium",
    "Growth Properties"
]

print("Current missing values in df:")
print(df[fill_targets].isna().sum(), "\n")

# Build merged versions once
df_m3 = df.merge(df3_small, on="COSMIC_ID", how="left")
df_m4 = df.merge(df4_small, on="COSMIC_ID", how="left")

# Check how much helper coverage exists specifically where df is missing
print("Coverage from df3 where df is missing:\n")
print("MSI -> msi_status:",
      df_m3.loc[df_m3["Microsatellite instability Status (MSI)"].isna(), "msi_status"].notna().sum())
print("Growth Properties -> growth_properties:",
      df_m3.loc[df_m3["Growth Properties"].isna(), "growth_properties"].notna().sum())
print("TCGA_DESC -> cancer_type:",
      df_m3.loc[df_m3["TCGA_DESC"].isna(), "cancer_type"].notna().sum())

print("\nCoverage from df4 where df is missing:\n")
print("Screen Medium -> OnboardedMedia:",
      df_m4.loc[df_m4["Screen Medium"].isna(), "OnboardedMedia"].notna().sum())
print("Growth Properties -> GrowthPattern:",
      df_m4.loc[df_m4["Growth Properties"].isna(), "GrowthPattern"].notna().sum())
print("TCGA_DESC -> OncotreePrimaryDisease:",
      df_m4.loc[df_m4["TCGA_DESC"].isna(), "OncotreePrimaryDisease"].notna().sum())

# Show only rows where df is missing but helper has a value
print("\nSample MSI fill candidates from df3:")
print(
    df_m3.loc[
        df_m3["Microsatellite instability Status (MSI)"].isna() &
        df_m3["msi_status"].notna(),
        ["COSMIC_ID", "CELL_LINE_NAME", "Microsatellite instability Status (MSI)", "msi_status"]
    ].head(10)
)

print("\nSample Growth Properties fill candidates from df3:")
print(
    df_m3.loc[
        df_m3["Growth Properties"].isna() &
        df_m3["growth_properties"].notna(),
        ["COSMIC_ID", "CELL_LINE_NAME", "Growth Properties", "growth_properties"]
    ].head(10)
)

print("\nSample Screen Medium fill candidates from df4:")
print(
    df_m4.loc[
        df_m4["Screen Medium"].isna() &
        df_m4["OnboardedMedia"].notna(),
        ["COSMIC_ID", "CELL_LINE_NAME", "Screen Medium", "OnboardedMedia"]
    ].head(10)
)

Current missing values in df:
TCGA_DESC                                   1067
GDSC Tissue descriptor 1                    9366
GDSC Tissue descriptor 2                    9366
Microsatellite instability Status (MSI)    12353
Screen Medium                               9366
Growth Properties                           9366
dtype: int64 

Coverage from df3 where df is missing:

MSI -> msi_status: 11723
Growth Properties -> growth_properties: 9366
TCGA_DESC -> cancer_type: 1067

Coverage from df4 where df is missing:

Screen Medium -> OnboardedMedia: 4464
Growth Properties -> GrowthPattern: 9366
TCGA_DESC -> OncotreePrimaryDisease: 1067

Sample MSI fill candidates from df3:
     COSMIC_ID CELL_LINE_NAME Microsatellite instability Status (MSI)  \
6       687452           5637                                     NaN   
62      713869       VMRC-LCD                                     NaN   
72      724812            T24                                     NaN   
114     753551         DMS-7

Output explanation - The output tells us whether the merge is structurally safe. If the row count stays the same before and after the merge, that usually means each row in df matched cleanly and the merge did not create duplicates. If the row count increases, that is a warning sign that some COSMIC_IDs are matching multiple rows in the helper table, which means you should not trust that merge yet.

Do the safe fills for MSI and Growth Properties (THIS IS PREPROCESSING; Nothing else for now until officially)
- This step is actually filling missing values in the main dataset, but only for fields that looked reliable enough to copy over directly. The reason MSI and Growth Properties were filled here is that the matching helper columns seemed closely aligned and low-risk compared to cancer/tissue labels. It works by merging those helper columns into a copy of df, then using .fillna() so only missing values are replaced while existing values stay untouched.

In [10]:
# Work on a copy
df_clean = df.copy()

# Merge in helper columns from df3
df_clean = df_clean.merge(
    df3_small[["COSMIC_ID", "msi_status", "growth_properties", "cancer_type"]],
    on="COSMIC_ID",
    how="left"
)

# Fill MSI from df3 where df is missing
df_clean["Microsatellite instability Status (MSI)"] = (
    df_clean["Microsatellite instability Status (MSI)"]
    .fillna(df_clean["msi_status"])
)

# Fill Growth Properties from df3 where df is missing
df_clean["Growth Properties"] = (
    df_clean["Growth Properties"]
    .fillna(df_clean["growth_properties"])
)

# Check how much missingness remains after safe fills
print("Remaining missing values after MSI and Growth fill:")
print(df_clean[[
    "Microsatellite instability Status (MSI)",
    "Growth Properties"
]].isna().sum())


Remaining missing values after MSI and Growth fill:
Microsatellite instability Status (MSI)    630
Growth Properties                            0
dtype: int64


Outcome explanation - The output tells you how much missingness is left after the fill. If the missing count drops a lot, that means the helper file successfully recovered many values. If the missing count barely changes, then either the helper file did not have much coverage or the matching between files was weak.

Only inspect TCGA fill candidates, not finalize TCGA preprocessing yet
- This step is trying to see whether df3 might help with missing TCGA_DESC, but without making the fill yet. The reason for being careful is that cancer_type in df3 may not match TCGA_DESC exactly, so filling too early could introduce wrong labels. It works by printing rows where TCGA_DESC is missing but cancer_type exists, and also comparing rows where both already exist to spot naming mismatches or inconsistencies.

In [11]:

# Compare TCGA_DESC with df3 cancer_type only on rows where TCGA_DESC is missing
tcga_candidates = df_clean.loc[
    df_clean["TCGA_DESC"].isna() & df_clean["cancer_type"].notna(),
    ["COSMIC_ID", "CELL_LINE_NAME", "TCGA_DESC", "cancer_type"]
].drop_duplicates()

print("\nSample TCGA fill candidates from df3:")
print(tcga_candidates.head(20))

# Also compare rows where TCGA_DESC already exists to see naming mismatch pattern
tcga_compare = df_clean.loc[
    df_clean["TCGA_DESC"].notna() & df_clean["cancer_type"].notna(),
    ["TCGA_DESC", "cancer_type"]
].drop_duplicates()

print("\nSample existing TCGA_DESC vs df3 cancer_type pairs:")
print(tcga_compare.head(30))

# Cleanup helper columns not yet needed as final columns
# (keep cancer_type for now because we still need to inspect TCGA mapping)
df_clean = df_clean.drop(columns=["msi_status", "growth_properties"])

print("\ndf_clean shape:", df_clean.shape)


Sample TCGA fill candidates from df3:
     COSMIC_ID CELL_LINE_NAME TCGA_DESC                cancer_type
221     906695       BONNA-12       NaN        Other Blood Cancers
383     908133     MHH-CALL-4       NaN   B-Lymphoblastic Leukemia
405     908441          NCCIT       NaN        Other Solid Cancers
922    1331055       NCI-H740       NaN  Small Cell Lung Carcinoma
947    1503373          U-CH2       NaN        Other Solid Cancers
961    1659929        SNU-283       NaN       Colorectal Carcinoma

Sample existing TCGA_DESC vs df3 cancer_type pairs:
        TCGA_DESC                         cancer_type
0              MB                 Other Solid Cancers
1    UNCLASSIFIED                     Ewing's Sarcoma
5            SKCM                            Melanoma
6            BLCA                   Bladder Carcinoma
10           CESC                  Cervical Carcinoma
13            GBM                        Glioblastoma
18            GBM                              Glioma
21   UN

Outcome explanation - The output tells you whether df3.cancer_type looks like a reasonable helper source for TCGA_DESC, but only as an inspection step. The candidate rows show where TCGA_DESC is missing but cancer_type exists, meaning those rows are possible fill opportunities. The comparison rows where both already exist tell you whether the naming styles line up well or whether there are mismatches, which helps decide if a direct fill would be safe or too risky.

Check whether df2 is safe enough to use for TCGA and tissue descriptor filling
- This step is trying to answer whether df2 can be trusted as a lookup source for TCGA_DESC, GDSC Tissue descriptor 1, and GDSC Tissue descriptor 2. The reason this comes next is that MSI and Growth were already safe enough to fill, but these cancer/tissue labels are more sensitive, so we need to make sure one COSMIC_ID does not point to conflicting descriptor values in df2. It works by grouping df2 by COSMIC_ID and counting how many unique values each ID has for the descriptor columns. If most COSMIC_IDs have only one value per field, then df2 is likely safe to collapse into a one-row-per-ID lookup later; if many IDs have multiple values, then we need to investigate more before using it.

In [12]:
# Validation Step 6: check whether df2 is internally consistent per COSMIC_ID
df2_check = df2.copy()

# Standardize key name if needed
if "COSMIC ID" in df2_check.columns:
    df2_check = df2_check.rename(columns={"COSMIC ID": "COSMIC_ID"})

# Keep only columns needed for TCGA / descriptor validation
cols_needed = ["COSMIC_ID", "TCGA Desc", "GDSC Desc1", "GDSC Desc2"]
df2_check = df2_check[[c for c in cols_needed if c in df2_check.columns]].copy()

# Clean COSMIC_ID
df2_check["COSMIC_ID"] = pd.to_numeric(df2_check["COSMIC_ID"], errors="coerce")

# Clean text columns
for col in ["TCGA Desc", "GDSC Desc1", "GDSC Desc2"]:
    if col in df2_check.columns:
        df2_check[col] = (
            df2_check[col]
            .astype("string")
            .str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        )

# Remove rows without COSMIC_ID
df2_check = df2_check[df2_check["COSMIC_ID"].notna()].copy()

# Count how many unique non-missing values each COSMIC_ID has
consistency_check = (
    df2_check.groupby("COSMIC_ID")[["TCGA Desc", "GDSC Desc1", "GDSC Desc2"]]
    .nunique(dropna=True)
    .reset_index()
)

print("Sample consistency check:")
print(consistency_check.head(20))

# IDs that look safe (at most 1 unique value in each descriptor field)
safe_ids = consistency_check[
    (consistency_check["TCGA Desc"] <= 1) &
    (consistency_check["GDSC Desc1"] <= 1) &
    (consistency_check["GDSC Desc2"] <= 1)
]

conflict_ids = consistency_check[
    (consistency_check["TCGA Desc"] > 1) |
    (consistency_check["GDSC Desc1"] > 1) |
    (consistency_check["GDSC Desc2"] > 1)
]

print("\nTotal COSMIC_IDs checked:", len(consistency_check))
print("Safe COSMIC_IDs:", len(safe_ids))
print("Conflicting COSMIC_IDs:", len(conflict_ids))

print("\nSample conflicting COSMIC_IDs:")
print(conflict_ids.head(20))

Sample consistency check:
    COSMIC_ID  TCGA Desc  GDSC Desc1  GDSC Desc2
0      683667          1           1           1
1      684052          1           1           1
2      684057          1           1           1
3      684059          1           1           1
4      684062          1           1           1
5      684072          1           1           1
6      687448          1           1           1
7      687452          1           1           1
8      687455          1           1           1
9      687457          1           1           1
10     687459          1           1           1
11     687505          1           1           1
12     687506          1           1           1
13     687514          1           1           1
14     687561          1           1           1
15     687562          1           1           1
16     687563          1           1           1
17     687568          1           1           1
18     687586          1           1       

Output explanation - It shows that for all 969 COSMIC_IDs checked, each one has at most one unique value for TCGA Desc, GDSC Desc1, and GDSC Desc2. That means df2 is internally consistent for these three columns, so it looks safe to collapse into a one-row-per-COSMIC_ID lookup table for later filling. The fact that Safe COSMIC_IDs = 969 and Conflicting COSMIC_IDs = 0 means there were no duplicate IDs with contradictory descriptor labels. So the validation goal here was successful: df2 passed the consistency test, and the next step can move from validation into building the safe lookup table from df2.

Build the safe df2 lookup and measure fill coverage
- Now that df2 passed the consistency test, the next step is to turn it into a clean one-row-per-COSMIC_ID lookup table and check how much missing data it could recover in df_clean. The reason for doing coverage first is the same as before: even if a source is safe, we still want to know whether it actually helps enough before using it for the actual fill. This works by collapsing df2 so each COSMIC_ID has one value for TCGA Desc, GDSC Desc1, and GDSC Desc2, then merging that lookup into a temporary validation table. After that, it counts how many rows in df_clean are missing each target field and how many of those missing rows have a usable value available from df2.

In [13]:
# Validation Step 7: build safe df2 lookup and check fill coverage

# Start from the cleaned validation copy from the previous step
df2_lookup = df2_check.copy()

# Collapse to one row per COSMIC_ID
def first_non_null(series):
    non_null = series.dropna()
    return non_null.iloc[0] if len(non_null) > 0 else pd.NA

df2_lookup = (
    df2_lookup.groupby("COSMIC_ID", as_index=False)
    .agg({
        "TCGA Desc": first_non_null,
        "GDSC Desc1": first_non_null,
        "GDSC Desc2": first_non_null
    })
)

print("df2_lookup shape:", df2_lookup.shape)
print(df2_lookup.head(10))

# Merge lookup into a temporary validation table
df_val = df_clean.merge(
    df2_lookup.rename(columns={
        "TCGA Desc": "TCGA_DESC_df2",
        "GDSC Desc1": "GDSC_Tissue_descriptor_1_df2",
        "GDSC Desc2": "GDSC_Tissue_descriptor_2_df2"
    }),
    on="COSMIC_ID",
    how="left"
)

# Check current missingness in df_clean
print("\nCurrent missing values in df_clean:")
print(df_clean[[
    "TCGA_DESC",
    "GDSC Tissue descriptor 1",
    "GDSC Tissue descriptor 2"
]].isna().sum())

# Check coverage from df2 lookup where df_clean is missing
print("\nCoverage from df2 lookup where df_clean is missing:")
print("TCGA_DESC -> TCGA Desc:",
      df_val.loc[df_val["TCGA_DESC"].isna(), "TCGA_DESC_df2"].notna().sum())
print("GDSC Tissue descriptor 1 -> GDSC Desc1:",
      df_val.loc[df_val["GDSC Tissue descriptor 1"].isna(), "GDSC_Tissue_descriptor_1_df2"].notna().sum())
print("GDSC Tissue descriptor 2 -> GDSC Desc2:",
      df_val.loc[df_val["GDSC Tissue descriptor 2"].isna(), "GDSC_Tissue_descriptor_2_df2"].notna().sum())

# Show sample fill candidates
print("\nSample TCGA fill candidates from df2:")
print(
    df_val.loc[
        df_val["TCGA_DESC"].isna() & df_val["TCGA_DESC_df2"].notna(),
        ["COSMIC_ID", "CELL_LINE_NAME", "TCGA_DESC", "TCGA_DESC_df2"]
    ].drop_duplicates().head(20)
)

print("\nSample descriptor 1 fill candidates from df2:")
print(
    df_val.loc[
        df_val["GDSC Tissue descriptor 1"].isna() & df_val["GDSC_Tissue_descriptor_1_df2"].notna(),
        ["COSMIC_ID", "CELL_LINE_NAME", "GDSC Tissue descriptor 1", "GDSC_Tissue_descriptor_1_df2"]
    ].drop_duplicates().head(20)
)

print("\nSample descriptor 2 fill candidates from df2:")
print(
    df_val.loc[
        df_val["GDSC Tissue descriptor 2"].isna() & df_val["GDSC_Tissue_descriptor_2_df2"].notna(),
        ["COSMIC_ID", "CELL_LINE_NAME", "GDSC Tissue descriptor 2", "GDSC_Tissue_descriptor_2_df2"]
    ].drop_duplicates().head(20)
)

df2_lookup shape: (969, 4)
   COSMIC_ID     TCGA Desc         GDSC Desc1        GDSC Desc2
0     683667            MB     nervous_system   medulloblastoma
1     684052  UNCLASSIFIED        soft_tissue  rhabdomyosarcoma
2     684057  UNCLASSIFIED               bone    ewings_sarcoma
3     684059  UNCLASSIFIED               bone    ewings_sarcoma
4     684062  UNCLASSIFIED               bone    ewings_sarcoma
5     684072  UNCLASSIFIED               bone    ewings_sarcoma
6     687448          SKCM               skin          melanoma
7     687452          BLCA  urogenital_system           bladder
8     687455          BLCA  urogenital_system           bladder
9     687457          BLCA  urogenital_system           bladder

Current missing values in df_clean:
TCGA_DESC                   1067
GDSC Tissue descriptor 1    9366
GDSC Tissue descriptor 2    9366
dtype: int64

Coverage from df2 lookup where df_clean is missing:
TCGA_DESC -> TCGA Desc: 0
GDSC Tissue descriptor 1 -> GDSC Desc1: 9

Output explanation - For this output, the main takeaway is very clear: the df2 lookup worked structurally and has full recovery power for both descriptor columns, but no recovery power for TCGA_DESC. The coverage counts show 0 for TCGA_DESC, which means none of the rows missing TCGA_DESC in df_clean have a usable TCGA Desc coming from df2; in contrast, both tissue descriptor columns have coverage for all 9366 missing rows, which is an excellent sign.

Check label alignment between df_clean and df2 for the descriptor columns
- Before filling GDSC Tissue descriptor 1 and GDSC Tissue descriptor 2, this step checks whether the labels already present in df_clean match the naming style coming from df2. The reason is that even if df2 has perfect coverage, we still want to confirm we are not mixing two incompatible label systems. This works by looking only at rows where both sides already have non-missing values, then comparing df_clean’s descriptor columns against the corresponding values from df2. If most pairs match exactly or look clearly equivalent, then the fill is safe to do next; if there are many mismatches, we may need a small harmonization step first.

In [14]:
# Validation Step 8: compare existing descriptor labels in df_clean vs df2 lookup

# Reuse df_val from the previous step
desc1_compare = (
    df_val.loc[
        df_val["GDSC Tissue descriptor 1"].notna() &
        df_val["GDSC_Tissue_descriptor_1_df2"].notna(),
        ["GDSC Tissue descriptor 1", "GDSC_Tissue_descriptor_1_df2"]
    ]
    .drop_duplicates()
    .sort_values(["GDSC Tissue descriptor 1", "GDSC_Tissue_descriptor_1_df2"])
)

desc2_compare = (
    df_val.loc[
        df_val["GDSC Tissue descriptor 2"].notna() &
        df_val["GDSC_Tissue_descriptor_2_df2"].notna(),
        ["GDSC Tissue descriptor 2", "GDSC_Tissue_descriptor_2_df2"]
    ]
    .drop_duplicates()
    .sort_values(["GDSC Tissue descriptor 2", "GDSC_Tissue_descriptor_2_df2"])
)

print("Sample existing descriptor 1 vs df2 pairs:")
print(desc1_compare.head(30))

print("\nSample existing descriptor 2 vs df2 pairs:")
print(desc2_compare.head(30))

# Optional exact-match summary
desc1_exact = (
    df_val.loc[
        df_val["GDSC Tissue descriptor 1"].notna() &
        df_val["GDSC_Tissue_descriptor_1_df2"].notna()
    ]
    .assign(match=lambda x: x["GDSC Tissue descriptor 1"] == x["GDSC_Tissue_descriptor_1_df2"])
)

desc2_exact = (
    df_val.loc[
        df_val["GDSC Tissue descriptor 2"].notna() &
        df_val["GDSC_Tissue_descriptor_2_df2"].notna()
    ]
    .assign(match=lambda x: x["GDSC Tissue descriptor 2"] == x["GDSC_Tissue_descriptor_2_df2"])
)

print("\nDescriptor 1 exact matches:",
      int(desc1_exact["match"].sum()), "/", len(desc1_exact))

print("Descriptor 2 exact matches:",
      int(desc2_exact["match"].sum()), "/", len(desc2_exact))

Sample existing descriptor 1 vs df2 pairs:
    GDSC Tissue descriptor 1 GDSC_Tissue_descriptor_1_df2
90            aero_dig_tract         aero_digestive_tract
1                       bone                         bone
91                    breast                       breast
226         digestive_system             digestive_system
102                   kidney                       kidney
168          large_intestine             digestive_system
112                 leukemia                        blood
58                      lung                         lung
21                lung_NSCLC                         lung
40                 lung_SCLC                         lung
196                 lymphoma                        blood
73                   myeloma                        blood
0             nervous_system               nervous_system
59             neuroblastoma               nervous_system
84                  pancreas                     pancreas
5                       skin 

# Preprocessing
- MSI and growth properties filled the missing values above

Fill GDSC Tissue descriptor 2
- This step fills the missing values in GDSC Tissue descriptor 2 in df_clean using the validated one-row-per-COSMIC_ID lookup from df2. We are doing this because validation showed that df2 is internally consistent, has complete coverage for the missing rows in this column, and its labels align very well with the existing values in the main dataset. The code merges the descriptor 2 helper column from the safe df2 lookup into df_clean, creates a flag showing which rows were filled, and then uses .fillna() so only missing values are replaced. After that, it prints how many values were filled and how many missing values remain.

In [15]:
# Preprocessing Step: Fill GDSC Tissue descriptor 2

# Merge descriptor 2 helper column into df_clean
df_clean = df_clean.merge(
    df2_lookup[["COSMIC_ID", "GDSC Desc2"]].rename(columns={
        "GDSC Desc2": "GDSC_Tissue_descriptor_2_df2"
    }),
    on="COSMIC_ID",
    how="left"
)

# Create fill flag before filling
df_clean["descriptor2_was_imputed"] = (
    df_clean["GDSC Tissue descriptor 2"].isna() &
    df_clean["GDSC_Tissue_descriptor_2_df2"].notna()
)

# Fill only missing values
df_clean["GDSC Tissue descriptor 2"] = (
    df_clean["GDSC Tissue descriptor 2"]
    .fillna(df_clean["GDSC_Tissue_descriptor_2_df2"])
)


In [16]:
'''
# Save checkpoint after GDSC Tissue descriptor 2 fill
df_clean.to_csv("checkpoint_after_tissue2_fill.csv", index=False)
'''

'\n# Save checkpoint after GDSC Tissue descriptor 2 fill\ndf_clean.to_csv("checkpoint_after_tissue2_fill.csv", index=False)\n'

Fill GDSC Tissue descriptor 1
- This step fills missing values in GDSC Tissue descriptor 1 using the safe one-row-per-COSMIC_ID lookup built from df2. We are doing this because validation showed df2 was internally consistent, had complete coverage for the missing rows in this column, and the labels were still compatible enough to use, even though some are broader tissue groups.
- The code merges the descriptor 1 helper column from df2_lookup into df_clean, creates a flag for rows that get filled, and then uses .fillna() so only missing values are replaced. After that, it prints how many rows were filled, how many missing values remain, and a sample of the filled rows so you can inspect them.

In [17]:
# Preprocessing Step: Fill GDSC Tissue descriptor 1

# Merge descriptor 1 helper column into df_clean
df_clean = df_clean.merge(
    df2_lookup[["COSMIC_ID", "GDSC Desc1"]].rename(columns={
        "GDSC Desc1": "GDSC_Tissue_descriptor_1_df2"
    }),
    on="COSMIC_ID",
    how="left"
)

# Create fill flag before filling
df_clean["descriptor1_was_imputed"] = (
    df_clean["GDSC Tissue descriptor 1"].isna() &
    df_clean["GDSC_Tissue_descriptor_1_df2"].notna()
)

# Fill only missing values
df_clean["GDSC Tissue descriptor 1"] = (
    df_clean["GDSC Tissue descriptor 1"]
    .fillna(df_clean["GDSC_Tissue_descriptor_1_df2"])
)
'''
# Check results
print("Rows filled for GDSC Tissue descriptor 1:",
      int(df_clean["descriptor1_was_imputed"].sum()))

print("\nRemaining missing values in GDSC Tissue descriptor 1:")
print(df_clean["GDSC Tissue descriptor 1"].isna().sum())

print("\nSample filled rows:")
print(
    df_clean.loc[
        df_clean["descriptor1_was_imputed"],
        ["COSMIC_ID", "CELL_LINE_NAME", "GDSC Tissue descriptor 1", "GDSC_Tissue_descriptor_1_df2"]
    ].drop_duplicates().head(20)
) '''

'\n# Check results\nprint("Rows filled for GDSC Tissue descriptor 1:",\n      int(df_clean["descriptor1_was_imputed"].sum()))\n\nprint("\nRemaining missing values in GDSC Tissue descriptor 1:")\nprint(df_clean["GDSC Tissue descriptor 1"].isna().sum())\n\nprint("\nSample filled rows:")\nprint(\n    df_clean.loc[\n        df_clean["descriptor1_was_imputed"],\n        ["COSMIC_ID", "CELL_LINE_NAME", "GDSC Tissue descriptor 1", "GDSC_Tissue_descriptor_1_df2"]\n    ].drop_duplicates().head(20)\n) '

Clean selected feature columns
- This step cleans selected columns in df_clean by standardizing text formatting and converting unusable blanks into proper missing values. We are doing this now because once you move into feature selection, encoding, and modeling, inconsistent text values can split one real category into several fake ones, which weakens the model and makes interpretation harder. The code trims whitespace from selected text columns, converts empty strings to missing values, and prints quick summaries so you can see what changed. 
- We would start with a conservative cleanup on the most likely feature columns first: TARGET, DRUG_NAME, TCGA_DESC, Microsatellite instability Status (MSI), Growth Properties, GDSC Tissue descriptor 1, and GDSC Tissue descriptor 2.

In [18]:
# Columns to clean if they exist
text_cols_to_clean = [
    "TARGET",
    "DRUG_NAME",
    "TCGA_DESC",
    "Microsatellite instability Status (MSI)",
    "Growth Properties",
    "GDSC Tissue descriptor 1",
    "GDSC Tissue descriptor 2"
]

# Keep only columns that actually exist in df_clean
text_cols_to_clean = [col for col in text_cols_to_clean if col in df_clean.columns]

# Check missing counts before cleaning
print("Missing values before cleaning:")
print(df_clean[text_cols_to_clean].isna().sum())

# Clean text columns
for col in text_cols_to_clean:
    df_clean[col] = (
        df_clean[col]
        .astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            " ": pd.NA,
            "nan": pd.NA,
            "None": pd.NA,
            "NONE": pd.NA,
            "null": pd.NA,
            "NULL": pd.NA
        })
    )

# Check missing counts after cleaning
print("\nMissing values after cleaning:")
print(df_clean[text_cols_to_clean].isna().sum())

# Show sample unique values for key columns
for col in ["TARGET", "DRUG_NAME", "Microsatellite instability Status (MSI)", "Growth Properties"]:
    if col in df_clean.columns:
        print(f"\nSample cleaned values in {col}:")
        print(df_clean[col].dropna().drop_duplicates().sort_values().head(20).tolist())

Missing values before cleaning:
TARGET                                     27155
DRUG_NAME                                      0
TCGA_DESC                                   1067
Microsatellite instability Status (MSI)      630
Growth Properties                              0
GDSC Tissue descriptor 1                       0
GDSC Tissue descriptor 2                       0
dtype: int64

Missing values after cleaning:
TARGET                                     27872
DRUG_NAME                                      0
TCGA_DESC                                   1067
Microsatellite instability Status (MSI)      630
Growth Properties                              0
GDSC Tissue descriptor 1                       0
GDSC Tissue descriptor 2                       0
dtype: int64

Sample cleaned values in TARGET:
['ABL', 'ABL, SRC, Ephrins, PDGFR, KIT', 'ADRA1A, ADRB1', 'AKT1', 'AKT1, AKT, AKT3', 'AKT1, AKT2', 'AKT1, AKT2, AKT3', 'AKT1, AKT2, AKT3, ROCK2', 'AR', 'ATM', 'ATR', 'AURKA', 'AURKA, AURKB',

The cleaning step did not create new missing values. Instead, it revealed hidden missing values in TARGET by converting blank or placeholder text entries into proper missing values.

In [19]:
# Preprocessing Step: Inspect current TCGA_DESC state before remapping

print("Current missing values in TCGA_DESC:")
print(df_clean["TCGA_DESC"].isna().sum())

print("\nTop TCGA_DESC categories:")
print(df_clean["TCGA_DESC"].value_counts(dropna=False).head(30))

# Optional: check specifically for unresolved labels
if "UNCLASSIFIED" in df_clean["TCGA_DESC"].astype("string").dropna().unique():
    print("\nCount of UNCLASSIFIED:")
    print((df_clean["TCGA_DESC"] == "UNCLASSIFIED").sum())

Current missing values in TCGA_DESC:
1067

Top TCGA_DESC categories:
TCGA_DESC
UNCLASSIFIED    45690
LUAD            15653
SCLC            13570
BRCA            13106
SKCM            12637
COREAD          12538
HNSC             9358
ESCA             9126
GBM              8384
OV               8166
DLBC             7978
PAAD             7513
NB               7469
KIRC             7462
ALL              6795
LAML             6209
STAD             6060
MESO             5561
BLCA             4724
MM               4598
LIHC             4164
THCA             4037
LUSC             3863
CESC             3811
LGG              3617
LCML             2611
UCEC             2512
PRAD             1676
MB               1072
<NA>             1067
Name: count, dtype: Int64

Count of UNCLASSIFIED:
45690


TCGA; Fill only the truly missing TCGA_DESC rows - 
- This step fills only rows where TCGA_DESC is <NA>, using a staged rule-based process. It is done first because truly missing values should be recovered before broader remapping steps like OTHER → PRAD or UNCLASSIFIED cleanup. The code first harmonizes the cancer-type helper labels to match the TCGA naming style, then fills missing TCGA_DESC from that cancer-type column when available. After that, it applies manually verified cell-line remaps for specific cases that were already reviewed, and finally checks whether GDSC Tissue descriptor 2 can provide a fill only when that descriptor maps uniquely to a single TCGA label in the existing dataset.


In [20]:
print([c for c in df_clean.columns if "Cancer Type" in c or "cancer" in c.lower()])

['Cancer Type (matching TCGA label)', 'cancer_type']


In [21]:
# Preprocessing Step: TCGA Stage 1 — safer fill for truly missing TCGA_DESC
# Order:
# 1) cancer-type helper column
# 2) manual cell-line remaps
# 3) Tissue descriptor 2 only when it maps cleanly to one TCGA label in df_clean

# Checkpoint: rows that were truly missing before Stage 1
df_clean["tcga_missing_before_stage1"] = df_clean["TCGA_DESC"].isna()

tcga_missing_before = int(df_clean["TCGA_DESC"].isna().sum())
print("Missing TCGA_DESC before Stage 1 fill:", tcga_missing_before)

# Pass 1: fill missing TCGA_DESC from validated cancer-type column
CANCER_TYPE_COL = "Cancer Type (matching TCGA label)"

if CANCER_TYPE_COL in df_clean.columns:
    # Harmonize helper labels to TCGA-style labels
    df_clean[CANCER_TYPE_COL] = (
        df_clean[CANCER_TYPE_COL]
        .astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "COAD/READ": "COREAD",
            "UNABLE TO CLASSIFY": "UNCLASSIFIED",
            "nan": pd.NA,
            "None": pd.NA
        })
    )

    df_clean["tcga_filled_from_cancer_type"] = (
        df_clean["TCGA_DESC"].isna() & df_clean[CANCER_TYPE_COL].notna()
    )

    df_clean["TCGA_DESC"] = df_clean["TCGA_DESC"].fillna(df_clean[CANCER_TYPE_COL])

else:
    df_clean["tcga_filled_from_cancer_type"] = False
    print(f"Column '{CANCER_TYPE_COL}' not found — skipping Pass 1.")

# Pass 2: manual cell-line remaps from previous preprocessing
cell_line_remap = {
    "MHH-CALL-4": "ALL",
    "BONNA-12": "UNCLASSIFIED",
    "U-CH2": "UNCLASSIFIED",
    "NCCIT": "UNCLASSIFIED",
}

df_clean["tcga_filled_manually"] = False

for cell_line, new_label in cell_line_remap.items():
    mask = (
        (df_clean["CELL_LINE_NAME"] == cell_line) &
        (df_clean["tcga_missing_before_stage1"]) &
        (df_clean["TCGA_DESC"].isna())
    )
    df_clean.loc[mask, "TCGA_DESC"] = new_label
    df_clean.loc[mask, "tcga_filled_manually"] = True

# Pass 3: Tissue descriptor 2 -> TCGA fill, but ONLY if mapping is unique
# Learn mapping from existing non-missing TCGA rows inside df_clean itself
t2_col = "GDSC Tissue descriptor 2"

# Build mapping only from rows where both Tissue 2 and TCGA are known
t2_known = df_clean.loc[
    df_clean[t2_col].notna() & df_clean["TCGA_DESC"].notna(),
    [t2_col, "TCGA_DESC"]
].copy()

# Count distinct TCGA labels per Tissue 2
t2_uniques = (
    t2_known.groupby(t2_col)["TCGA_DESC"]
    .nunique()
    .reset_index(name="n_tcga")
)

# Keep only Tissue 2 values that map to exactly ONE TCGA label
t2_unique_only = t2_uniques.loc[t2_uniques["n_tcga"] == 1, t2_col]

t2_to_tcga_map = (
    t2_known[t2_known[t2_col].isin(t2_unique_only)]
    .drop_duplicates(subset=[t2_col, "TCGA_DESC"])
    .set_index(t2_col)["TCGA_DESC"]
    .to_dict()
)

# Store the inferred TCGA from unique Tissue 2 mapping
df_clean["tcga_from_desc2_unique"] = df_clean[t2_col].map(t2_to_tcga_map)

# Fill only rows that were originally missing and are still missing now
df_clean["tcga_filled_from_desc2"] = (
    df_clean["tcga_missing_before_stage1"] &
    df_clean["TCGA_DESC"].isna() &
    df_clean["tcga_from_desc2_unique"].notna()
)

df_clean.loc[
    df_clean["tcga_filled_from_desc2"],
    "TCGA_DESC"
] = df_clean.loc[
    df_clean["tcga_filled_from_desc2"],
    "tcga_from_desc2_unique"
]

# Results
tcga_missing_after = int(df_clean["TCGA_DESC"].isna().sum())

print("\nStage 1 fill results:")
print("Missing before:", tcga_missing_before)
print("Filled from cancer-type column:", int(df_clean["tcga_filled_from_cancer_type"].sum()))
print("Filled manually:", int(df_clean["tcga_filled_manually"].sum()))
print("Filled from Tissue descriptor 2 (unique mapping only):", int(df_clean["tcga_filled_from_desc2"].sum()))
print("Missing after:", tcga_missing_after)

print("\nNumber of unique Tissue descriptor 2 -> TCGA mappings used:")
print(len(t2_to_tcga_map))

print("\nSample Stage 1 filled rows:")
print(
    df_clean.loc[
        df_clean["tcga_missing_before_stage1"] & df_clean["TCGA_DESC"].notna(),
        ["COSMIC_ID", "CELL_LINE_NAME", "GDSC Tissue descriptor 2", "TCGA_DESC"]
    ].drop_duplicates().head(20)
)

Missing TCGA_DESC before Stage 1 fill: 1067

Stage 1 fill results:
Missing before: 1067
Filled from cancer-type column: 360
Filled manually: 707
Filled from Tissue descriptor 2 (unique mapping only): 0
Missing after: 0

Number of unique Tissue descriptor 2 -> TCGA mappings used:
39

Sample Stage 1 filled rows:
     COSMIC_ID CELL_LINE_NAME   GDSC Tissue descriptor 2     TCGA_DESC
221     906695       BONNA-12       hairy_cell_leukaemia  UNCLASSIFIED
383     908133     MHH-CALL-4            B_cell_leukemia           ALL
405     908441          NCCIT                     testis  UNCLASSIFIED
922    1331055       NCI-H740  lung_small_cell_carcinoma          SCLC
947    1503373          U-CH2                 bone_other  UNCLASSIFIED
961    1659929        SNU-283            large_intestine        COREAD


In [22]:
'''
# Save checkpoint after GDSC Tissue descriptor 2 fill
df_clean.to_csv("checkpoint_after_tcga_missing.csv", index=False)
 '''

'\n# Save checkpoint after GDSC Tissue descriptor 2 fill\ndf_clean.to_csv("checkpoint_after_tcga_missing.csv", index=False)\n '

TCGA; Remap all the “OTHER” into PRAD because they are all “urogenital_system” & “prostate” 
- This step checks only rows where TCGA_DESC is currently labeled OTHER and applies a rule-based remap to PRAD when those rows match the prostate-related criteria used in the previous preprocessing. It is done after the true missing-value fill because OTHER is not a missing value, but a broader category that needs a separate remapping decision.


In [23]:
# Preprocessing Step: TCGA Stage 2 — OTHER -> PRAD only

other_before = int((df_clean["TCGA_DESC"] == "OTHER").sum())
prad_before = int((df_clean["TCGA_DESC"] == "PRAD").sum())

print("Before Stage 2:")
print("OTHER rows:", other_before)
print("PRAD rows:", prad_before)

# Track rows changed in this stage
df_clean["tcga_other_to_prad"] = df_clean["TCGA_DESC"] == "OTHER"

# Bulk remap OTHER -> PRAD
df_clean["TCGA_DESC"] = df_clean["TCGA_DESC"].replace({"OTHER": "PRAD"})

other_after = int((df_clean["TCGA_DESC"] == "OTHER").sum())
prad_after = int((df_clean["TCGA_DESC"] == "PRAD").sum())

print("\nStage 2 results:")
print("Rows changed OTHER -> PRAD:", int(df_clean["tcga_other_to_prad"].sum()))
print("OTHER remaining:", other_after)
print("PRAD rows now:", prad_after)

print("\nSample affected rows:")
print(
    df_clean.loc[
        df_clean["tcga_other_to_prad"],
        ["COSMIC_ID", "CELL_LINE_NAME", "TCGA_DESC"]
    ].drop_duplicates().head(20)
)

Before Stage 2:
OTHER rows: 179
PRAD rows: 1676

Stage 2 results:
Rows changed OTHER -> PRAD: 179
OTHER remaining: 0
PRAD rows now: 1855

Sample affected rows:
     COSMIC_ID CELL_LINE_NAME TCGA_DESC
897    1330975       NCI-H660      PRAD


In [24]:
# Validation Step: inspect UNCLASSIFIED rows against supplementary files by COSMIC_ID

import pandas as pd

# -----------------------------
# 1) Prepare UNCLASSIFIED slice
# -----------------------------
unclassified_check = df_clean.loc[
    df_clean["TCGA_DESC"] == "UNCLASSIFIED",
    ["COSMIC_ID", "CELL_LINE_NAME", "TCGA_DESC"]
].copy()

print("UNCLASSIFIED rows in df_clean:", len(unclassified_check))
print("Unique COSMIC_IDs in UNCLASSIFIED rows:", unclassified_check["COSMIC_ID"].nunique())

# -----------------------------
# 2) Standardize COSMIC_ID in helper files
# -----------------------------
# Adjust names only if needed depending on your notebook variable names
for helper_df_name in ["df2", "df3", "df4", "df5"]:
    if helper_df_name in globals():
        helper_df = globals()[helper_df_name].copy()
        if "COSMIC ID" in helper_df.columns and "COSMIC_ID" not in helper_df.columns:
            helper_df = helper_df.rename(columns={"COSMIC ID": "COSMIC_ID"})
        if "COSMIC_ID" in helper_df.columns:
            helper_df["COSMIC_ID"] = pd.to_numeric(helper_df["COSMIC_ID"], errors="coerce")
        globals()[helper_df_name + "_std"] = helper_df

# If your notebook uses different names, fallback aliases:
# df2 -> GDSC_DATASET.csv
# df3 -> genomic_features.csv
# df4 -> Model.csv
# df5 -> model_list_latest.csv

# -----------------------------
# 3) Build reduced helper tables with likely label columns
# -----------------------------
helper_tables = {}

if "df2_std" in globals():
    cols = [c for c in [
        "COSMIC_ID", "TCGA_DESC", "Cancer Type (matching TCGA label)",
        "cancer_type", "TCGA Desc"
    ] if c in df2_std.columns]
    if cols:
        helper_tables["GDSC_DATASET"] = df2_std[cols].drop_duplicates()

if "df3_std" in globals():
    cols = [c for c in [
        "COSMIC_ID", "TCGA Desc", "cancer_type", "GDSC Desc1", "GDSC Desc2"
    ] if c in df3_std.columns]
    if cols:
        helper_tables["genomic_features"] = df3_std[cols].drop_duplicates()

if "df4_std" in globals():
    cols = [c for c in [
        "COSMIC_ID", "TCGA_DESC", "Cancer Type (matching TCGA label)",
        "OncotreeLineage", "OncotreePrimaryDisease", "OncotreeSubtype"
    ] if c in df4_std.columns]
    if cols:
        helper_tables["Model"] = df4_std[cols].drop_duplicates()

if "df5_std" in globals():
    cols = [c for c in [
        "COSMIC_ID", "TCGA_DESC", "Cancer Type (matching TCGA label)",
        "OncotreeLineage", "OncotreePrimaryDisease", "OncotreeSubtype"
    ] if c in df5_std.columns]
    if cols:
        helper_tables["model_list_latest"] = df5_std[cols].drop_duplicates()

print("\nHelper tables prepared:")
for name, tbl in helper_tables.items():
    print(name, tbl.shape, "columns:", tbl.columns.tolist())

# -----------------------------
# 4) Merge helper candidates onto UNCLASSIFIED rows
# -----------------------------
unclassified_candidates = unclassified_check.copy()

for name, tbl in helper_tables.items():
    rename_map = {
        col: f"{name}__{col}"
        for col in tbl.columns
        if col != "COSMIC_ID"
    }
    unclassified_candidates = unclassified_candidates.merge(
        tbl.rename(columns=rename_map),
        on="COSMIC_ID",
        how="left"
    )

print("\nMerged candidate table shape:", unclassified_candidates.shape)

# -----------------------------
# 5) Quick candidate summaries
# -----------------------------
candidate_cols = [c for c in unclassified_candidates.columns if c not in ["COSMIC_ID", "CELL_LINE_NAME", "TCGA_DESC"]]

print("\nNon-missing candidate counts among UNCLASSIFIED rows:")
for col in candidate_cols:
    non_missing = unclassified_candidates[col].notna().sum()
    print(f"{col}: {non_missing}")

# Also estimate non-UNCLASSIFIED style values for text columns
print("\nPotentially useful non-UNCLASSIFIED candidate counts:")
for col in candidate_cols:
    s = unclassified_candidates[col].astype("string")
    useful = s.notna() & ~s.isin(["UNCLASSIFIED", "OTHER", "nan", "None", ""])
    print(f"{col}: {int(useful.sum())}")

# -----------------------------
# 6) Save checkpoint CSV for review
# -----------------------------
checkpoint_name = "checkpoint_unclassified_cosmicid_crosscheck.csv"
unclassified_candidates.to_csv(checkpoint_name, index=False)

print(f"\nCheckpoint saved as: {checkpoint_name}")

# -----------------------------
# 7) Optional small sample for quick viewing
# -----------------------------
print("\nSample rows with candidate info:")
display_cols = ["COSMIC_ID", "CELL_LINE_NAME", "TCGA_DESC"] + candidate_cols[:8]
print(unclassified_candidates[display_cols].head(20))

UNCLASSIFIED rows in df_clean: 46218
Unique COSMIC_IDs in UNCLASSIFIED rows: 184

Helper tables prepared:
GDSC_DATASET (969, 2) columns: ['COSMIC_ID', 'TCGA Desc']
genomic_features (1166, 2) columns: ['COSMIC_ID', 'cancer_type']
Model (1189, 4) columns: ['COSMIC_ID', 'OncotreeLineage', 'OncotreePrimaryDisease', 'OncotreeSubtype']

Merged candidate table shape: (46218, 8)

Non-missing candidate counts among UNCLASSIFIED rows:
GDSC_DATASET__TCGA Desc: 45690
genomic_features__cancer_type: 46218
Model__OncotreeLineage: 45210
Model__OncotreePrimaryDisease: 45210
Model__OncotreeSubtype: 45210

Potentially useful non-UNCLASSIFIED candidate counts:
GDSC_DATASET__TCGA Desc: 0
genomic_features__cancer_type: 46218
Model__OncotreeLineage: 45210
Model__OncotreePrimaryDisease: 45210
Model__OncotreeSubtype: 45210

Checkpoint saved as: checkpoint_unclassified_cosmicid_crosscheck.csv

Sample rows with candidate info:
    COSMIC_ID CELL_LINE_NAME     TCGA_DESC GDSC_DATASET__TCGA Desc  \
0      684057   

TCGA; Trying to reduce the “UNCLASSIFIED” TCGA value, so we have more accurate data actually to use  
- This step checks only rows where TCGA_DESC is currently UNCLASSIFIED and tries to replace them with a more specific TCGA label using helper information matched by COSMIC_ID. It is done after missing-value fill and OTHER → PRAD remap because UNCLASSIFIED is not blank, but still too broad to be useful. The code first gathers cancer-related helper labels from the supplementary files, then learns which helper labels map cleanly to known TCGA labels in the existing dataset, and finally updates UNCLASSIFIED only when the evidence leads to one clear TCGA category.


In [25]:
# Resolve UNCLASSIFIED using supplementary evidence + learned safe mappings

# Phase 1: Prepare helper evidence for UNCLASSIFIED COSMIC_IDs

stage3_before = int((df_clean["TCGA_DESC"] == "UNCLASSIFIED").sum())
print("UNCLASSIFIED rows before Stage 3:", stage3_before)

# Standardize helper files if they exist
for helper_df_name in ["df2", "df3", "df4", "df5"]:
    if helper_df_name in globals():
        helper_df = globals()[helper_df_name].copy()
        if "COSMIC ID" in helper_df.columns and "COSMIC_ID" not in helper_df.columns:
            helper_df = helper_df.rename(columns={"COSMIC ID": "COSMIC_ID"})
        if "COSMIC_ID" in helper_df.columns:
            helper_df["COSMIC_ID"] = pd.to_numeric(helper_df["COSMIC_ID"], errors="coerce")
        globals()[helper_df_name + "_std"] = helper_df

# Build reduced helper tables
helper_tables = {}

# df3 = genomic_features.csv
if "df3_std" in globals():
    cols = [c for c in [
        "COSMIC_ID", "cancer_type", "GDSC Desc1", "GDSC Desc2"
    ] if c in df3_std.columns]
    if cols:
        helper_tables["genomic_features"] = df3_std[cols].drop_duplicates()

# df4 = Model.csv
if "df4_std" in globals():
    cols = [c for c in [
        "COSMIC_ID", "OncotreeLineage", "OncotreePrimaryDisease", "OncotreeSubtype"
    ] if c in df4_std.columns]
    if cols:
        helper_tables["Model"] = df4_std[cols].drop_duplicates()

# df5 = model_list_latest.csv (optional)
if "df5_std" in globals():
    cols = [c for c in [
        "COSMIC_ID", "OncotreeLineage", "OncotreePrimaryDisease", "OncotreeSubtype"
    ] if c in df5_std.columns]
    if cols:
        helper_tables["model_list_latest"] = df5_std[cols].drop_duplicates()

print("\nHelper tables prepared:")
for name, tbl in helper_tables.items():
    print(name, tbl.shape, "columns:", tbl.columns.tolist())

# Unique COSMIC_ID-level table from df_clean
cosmic_master = (
    df_clean[["COSMIC_ID", "CELL_LINE_NAME", "TCGA_DESC"]]
    .drop_duplicates()
    .copy()
)

# Merge helper evidence onto the COSMIC master table
for name, tbl in helper_tables.items():
    rename_map = {
        col: f"{name}__{col}"
        for col in tbl.columns
        if col != "COSMIC_ID"
    }
    cosmic_master = cosmic_master.merge(
        tbl.rename(columns=rename_map),
        on="COSMIC_ID",
        how="left"
    )



UNCLASSIFIED rows before Stage 3: 46218

Helper tables prepared:
genomic_features (1166, 2) columns: ['COSMIC_ID', 'cancer_type']
Model (1189, 4) columns: ['COSMIC_ID', 'OncotreeLineage', 'OncotreePrimaryDisease', 'OncotreeSubtype']


In [26]:
# Phase 2: Learn safe helper-label -> TCGA mappings from already labeled (non-UNCLASSIFIED, non-missing) rows

known_master = cosmic_master.loc[
    cosmic_master["TCGA_DESC"].notna() &
    (cosmic_master["TCGA_DESC"] != "UNCLASSIFIED"),
].copy()

print("\nKnown labeled COSMIC_IDs available for mapping:",
      known_master["COSMIC_ID"].nunique())

mapping_sources = [
    "genomic_features__cancer_type",
    "Model__OncotreePrimaryDisease",
    "Model__OncotreeSubtype",
    "Model__OncotreeLineage",
    "model_list_latest__OncotreePrimaryDisease",
    "model_list_latest__OncotreeSubtype",
    "model_list_latest__OncotreeLineage",
]

mapping_sources = [c for c in mapping_sources if c in cosmic_master.columns]

def build_unique_mapping(df_source, source_col, target_col="TCGA_DESC"):
    tmp = df_source[[source_col, target_col]].dropna().copy()
    if tmp.empty:
        return {}
    tmp[source_col] = tmp[source_col].astype("string").str.strip()
    tmp[target_col] = tmp[target_col].astype("string").str.strip()

    # Exclude broad unresolved labels from training side
    tmp = tmp.loc[
        ~tmp[target_col].isin(["UNCLASSIFIED", "OTHER", "", "nan", "None"])
    ].copy()

    if tmp.empty:
        return {}

    counts = (
        tmp.groupby(source_col)[target_col]
        .nunique()
        .reset_index(name="n_tcga")
    )

    unique_keys = counts.loc[counts["n_tcga"] == 1, source_col]

    mapping = (
        tmp[tmp[source_col].isin(unique_keys)]
        .drop_duplicates(subset=[source_col, target_col])
        .set_index(source_col)[target_col]
        .to_dict()
    )
    return mapping

helper_to_tcga_maps = {}
for source_col in mapping_sources:
    helper_to_tcga_maps[source_col] = build_unique_mapping(known_master, source_col)

print("\nUnique helper-label -> TCGA mappings learned:")
for source_col, mp in helper_to_tcga_maps.items():
    print(f"{source_col}: {len(mp)}")

# Apply learned mappings to UNCLASSIFIED COSMIC_IDs

unclassified_master = cosmic_master.loc[
    cosmic_master["TCGA_DESC"] == "UNCLASSIFIED"
].copy()

print("\nUNCLASSIFIED unique COSMIC_IDs to evaluate:",
      unclassified_master["COSMIC_ID"].nunique())

# For each helper source, infer candidate TCGA using learned unique mapping
for source_col, mp in helper_to_tcga_maps.items():
    pred_col = f"{source_col}__pred_tcga"
    if source_col in unclassified_master.columns:
        unclassified_master[pred_col] = (
            unclassified_master[source_col]
            .astype("string")
            .str.strip()
            .map(mp)
        )

pred_cols = [c for c in unclassified_master.columns if c.endswith("__pred_tcga")]

# Build consensus across helper predictions
def consensus_tcga(row):
    vals = [v for v in row if pd.notna(v)]
    vals = [str(v).strip() for v in vals if str(v).strip() not in ["", "nan", "None"]]
    unique_vals = sorted(set(vals))
    if len(unique_vals) == 1:
        return unique_vals[0]
    return pd.NA

unclassified_master["stage3_consensus_tcga"] = unclassified_master[pred_cols].apply(consensus_tcga, axis=1)

# Optional: count how many helper predictions supported the final answer
def support_count(row):
    final_val = row["stage3_consensus_tcga"]
    if pd.isna(final_val):
        return 0
    vals = [row[c] for c in pred_cols if pd.notna(row[c])]
    return sum(str(v).strip() == str(final_val).strip() for v in vals)

unclassified_master["stage3_support_count"] = unclassified_master.apply(support_count, axis=1)

# Resolve only when there is a clear consensus label
resolved_cosmic = unclassified_master.loc[
    unclassified_master["stage3_consensus_tcga"].notna(),
    ["COSMIC_ID", "stage3_consensus_tcga", "stage3_support_count"]
].drop_duplicates()

print("\nUNCLASSIFIED COSMIC_IDs resolved by Stage 3:",
      resolved_cosmic["COSMIC_ID"].nunique())

# Merge resolved labels back to df_clean
df_clean = df_clean.merge(
    resolved_cosmic.rename(columns={"stage3_consensus_tcga": "stage3_resolved_tcga"}),
    on="COSMIC_ID",
    how="left"
)

df_clean["tcga_resolved_from_stage3"] = (
    (df_clean["TCGA_DESC"] == "UNCLASSIFIED") &
    df_clean["stage3_resolved_tcga"].notna()
)

df_clean.loc[
    df_clean["tcga_resolved_from_stage3"],
    "TCGA_DESC"
] = df_clean.loc[
    df_clean["tcga_resolved_from_stage3"],
    "stage3_resolved_tcga"
]

stage3_after = int((df_clean["TCGA_DESC"] == "UNCLASSIFIED").sum())




Known labeled COSMIC_IDs available for mapping: 785

Unique helper-label -> TCGA mappings learned:
genomic_features__cancer_type: 21
Model__OncotreePrimaryDisease: 33
Model__OncotreeSubtype: 84
Model__OncotreeLineage: 13

UNCLASSIFIED unique COSMIC_IDs to evaluate: 184

UNCLASSIFIED COSMIC_IDs resolved by Stage 3: 96


In [27]:
''' # Save checkpoint after GDSC Tissue descriptor 2 fill
df_clean.to_csv("checkpoint_after_UNCLASSIFIED_REMAP.csv", index=False)'''

' # Save checkpoint after GDSC Tissue descriptor 2 fill\ndf_clean.to_csv("checkpoint_after_UNCLASSIFIED_REMAP.csv", index=False)'

Genomic Feature Checking (Do we need it?)
- This is a validation step before merging genomic features into df_clean. We are doing this now because genomic features can be important predictors, but only if the merge is structurally safe and the added columns are useful enough to keep. The code standardizes COSMIC_ID, checks duplicate behavior in genomic_features.csv, runs a safe merge test, reports how many columns would be added, and summarizes missingness/sparsity in the genomic feature columns. That will tell us whether the file is ready to integrate and whether we should keep all genomic columns or only a selected subset.


EDA BEFORE PRE

In [28]:
'''
"CHECKING FOR THE PIPELINE.xlsx"# Validation Step: check whether genomic_features.csv is ready to add as predictors

# 1) Prepare genomic features copy
gf = df3.copy()   # assuming df3 = genomic_features.csv

if "COSMIC ID" in gf.columns and "COSMIC_ID" not in gf.columns:
    gf = gf.rename(columns={"COSMIC ID": "COSMIC_ID"})

gf["COSMIC_ID"] = pd.to_numeric(gf["COSMIC_ID"], errors="coerce")

print("genomic_features shape:", gf.shape)
print("Unique COSMIC_IDs in genomic_features:", gf["COSMIC_ID"].nunique())
print("Rows with missing COSMIC_ID:", gf["COSMIC_ID"].isna().sum())

# ------------------------------------------------------------
# 2) Check duplicate COSMIC_ID behavior
# ------------------------------------------------------------
gf_dup_counts = gf["COSMIC_ID"].value_counts(dropna=False)
gf_dups = gf_dup_counts[gf_dup_counts > 1]

print("\nNumber of duplicated COSMIC_IDs in genomic_features:", len(gf_dups))
print("Sample duplicated COSMIC_IDs:")
print(gf_dups.head(20))

# ------------------------------------------------------------
# 3) Identify likely genomic feature columns
# Exclude metadata/helper columns
# ------------------------------------------------------------
exclude_cols = {
    "COSMIC_ID",
    "model_name",
    "tissue",
    "cancer_type",
    "cancer_type_detail",
    "growth_properties",
    "msi_status",
    "sample_site",
    "TCGA Desc",
    "GDSC Desc1",
    "GDSC Desc2"
}

genomic_feature_cols = [c for c in gf.columns if c not in exclude_cols]

print("\nNumber of candidate genomic feature columns:", len(genomic_feature_cols))
print("Sample genomic feature columns:")
print(genomic_feature_cols[:30])

# ------------------------------------------------------------
# 4) If duplicates exist, check whether genomic feature values
# are consistent within each COSMIC_ID
# ------------------------------------------------------------
if len(gf_dups) > 0:
    gf_check = gf[gf["COSMIC_ID"].isin(gf_dups.index)].copy()

    # Count unique values per COSMIC_ID across feature columns
    consistency_summary = (
        gf_check.groupby("COSMIC_ID")[genomic_feature_cols]
        .nunique(dropna=True)
    )

    # A COSMIC_ID is "consistent" if every feature has at most 1 non-missing value
    consistent_ids = consistency_summary.max(axis=1) <= 1
    print("\nDuplicated COSMIC_IDs that are fully consistent across genomic features:",
          int(consistent_ids.sum()), "/", len(consistent_ids))

    print("\nSample inconsistent duplicated COSMIC_IDs:")
    print(consistency_summary.loc[~consistent_ids].head(10))
else:
    print("\nNo duplicated COSMIC_IDs detected in genomic_features.")

# ------------------------------------------------------------
# 5) Build a safe one-row-per-COSMIC_ID version if possible
# ------------------------------------------------------------
def first_non_null(series):
    non_null = series.dropna()
    return non_null.iloc[0] if len(non_null) > 0 else pd.NA

gf_lookup = (
    gf.groupby("COSMIC_ID", as_index=False)[genomic_feature_cols]
    .agg(first_non_null)
)

print("\nCollapsed genomic feature lookup shape:", gf_lookup.shape)
print("Unique COSMIC_IDs in collapsed lookup:", gf_lookup["COSMIC_ID"].nunique())

# ------------------------------------------------------------
# 6) Safe merge test with df_clean
# ------------------------------------------------------------
before_rows = len(df_clean)

df_gf_test = df_clean.merge(
    gf_lookup,
    on="COSMIC_ID",
    how="left"
)

after_rows = len(df_gf_test)

print("\nSafe merge test:")
print("Rows before merge:", before_rows)
print("Rows after merge :", after_rows)

# ------------------------------------------------------------
# 7) Coverage and sparsity summary
# ------------------------------------------------------------
added_cols = [c for c in gf_lookup.columns if c != "COSMIC_ID"]

non_missing_counts = df_gf_test[added_cols].notna().sum().sort_values(ascending=False)
missing_pct = (df_gf_test[added_cols].isna().mean() * 100).sort_values()

print("\nTop 20 genomic feature columns by non-missing count:")
print(non_missing_counts.head(20))

print("\nTop 20 genomic feature columns with lowest missing %:")
print(missing_pct.head(20))

print("\nTop 20 genomic feature columns with highest missing %:")
print(missing_pct.tail(20))

# ------------------------------------------------------------
# 8) Basic data-type preview
# ------------------------------------------------------------
print("\nData types of candidate genomic feature columns:")
print(df_gf_test[added_cols].dtypes.value_counts())

# Optional: sample values from a few genomic columns
sample_cols = added_cols[:10]
print("\nSample values from first 10 genomic feature columns:")
for col in sample_cols:
    print(f"\n{col}:")
    print(df_gf_test[col].dropna().astype(str).value_counts().head(10)) '''

'\n"CHECKING FOR THE PIPELINE.xlsx"# Validation Step: check whether genomic_features.csv is ready to add as predictors\n\n# 1) Prepare genomic features copy\ngf = df3.copy()   # assuming df3 = genomic_features.csv\n\nif "COSMIC ID" in gf.columns and "COSMIC_ID" not in gf.columns:\n    gf = gf.rename(columns={"COSMIC ID": "COSMIC_ID"})\n\ngf["COSMIC_ID"] = pd.to_numeric(gf["COSMIC_ID"], errors="coerce")\n\nprint("genomic_features shape:", gf.shape)\nprint("Unique COSMIC_IDs in genomic_features:", gf["COSMIC_ID"].nunique())\nprint("Rows with missing COSMIC_ID:", gf["COSMIC_ID"].isna().sum())\n\n# ------------------------------------------------------------\n# 2) Check duplicate COSMIC_ID behavior\n# ------------------------------------------------------------\ngf_dup_counts = gf["COSMIC_ID"].value_counts(dropna=False)\ngf_dups = gf_dup_counts[gf_dup_counts > 1]\n\nprint("\nNumber of duplicated COSMIC_IDs in genomic_features:", len(gf_dups))\nprint("Sample duplicated COSMIC_IDs:")\nprint(

Screen genomic_features.csv for actually useful biological/genomic columns

In [29]:
# Validation Step: screen genomic_features.csv for actually useful biological/genomic columns

# 1) Prepare genomic features copy
gf = df3.copy()   # assuming df3 = genomic_features.csv

if "COSMIC ID" in gf.columns and "COSMIC_ID" not in gf.columns:
    gf = gf.rename(columns={"COSMIC ID": "COSMIC_ID"})

gf["COSMIC_ID"] = pd.to_numeric(gf["COSMIC_ID"], errors="coerce")

print("Original genomic_features shape:", gf.shape)
print("Original unique COSMIC_IDs:", gf["COSMIC_ID"].nunique())
print("Rows with missing COSMIC_ID:", gf["COSMIC_ID"].isna().sum())

# Keep only rows with valid COSMIC_ID
gf = gf[gf["COSMIC_ID"].notna()].copy()

print("\nAfter dropping missing COSMIC_ID rows:")
print("Shape:", gf.shape)
print("Unique COSMIC_IDs:", gf["COSMIC_ID"].nunique())

# 2) Keep only likely useful biological/genomic columns
priority_cols = [
    "COSMIC_ID",
    "ploidy_snp6",
    "ploidy_wes",
    "ploidy_wgs",
    "mutational_burden",
    "mlh1_expression_by_ihc",
    "mlh1_promoter_methylation_status",
    "msh2_expression_by_ihc",
    "pms2_expression_by_ihc",
    "msh6_expression_by_ihc",
    "braf_mutation_identified",
    "braf_expression_by_ihc",
    "pik3ca_mutation_identified",
    "pten_expression_by_ihc",
    "pten_mutation_identified",
    "kras_mutation_identified",
    "mismatch_repair_status"
]

priority_cols = [c for c in priority_cols if c in gf.columns]

gf_bio = gf[priority_cols].copy()

print("\nSelected biological/genomic columns:")
print(priority_cols)
print("Number of selected columns (including COSMIC_ID):", len(priority_cols))

# 3) Check duplicate COSMIC_ID behavior for selected columns
gf_dup_counts = gf_bio["COSMIC_ID"].value_counts(dropna=False)
gf_dups = gf_dup_counts[gf_dup_counts > 1]

print("\nNumber of duplicated valid COSMIC_IDs in selected genomic data:", len(gf_dups))

if len(gf_dups) > 0:
    gf_check = gf_bio[gf_bio["COSMIC_ID"].isin(gf_dups.index)].copy()

    selected_feature_cols = [c for c in gf_bio.columns if c != "COSMIC_ID"]

    consistency_summary = (
        gf_check.groupby("COSMIC_ID")[selected_feature_cols]
        .nunique(dropna=True)
    )

    consistent_ids = consistency_summary.max(axis=1) <= 1
    print("Duplicated COSMIC_IDs fully consistent across selected columns:",
          int(consistent_ids.sum()), "/", len(consistent_ids))

    print("\nSample inconsistent duplicated COSMIC_IDs:")
    print(consistency_summary.loc[~consistent_ids].head(10))
else:
    print("No duplicated valid COSMIC_IDs detected in selected genomic data.")

# 4) Collapse to one row per COSMIC_ID
def first_non_null(series):
    non_null = series.dropna()
    return non_null.iloc[0] if len(non_null) > 0 else pd.NA

selected_feature_cols = [c for c in gf_bio.columns if c != "COSMIC_ID"]

gf_bio_lookup = (
    gf_bio.groupby("COSMIC_ID", as_index=False)[selected_feature_cols]
    .agg(first_non_null)
)

print("\nCollapsed biological/genomic lookup shape:", gf_bio_lookup.shape)
print("Unique COSMIC_IDs in collapsed lookup:", gf_bio_lookup["COSMIC_ID"].nunique())

# 5) Safe merge test with df_clean
# ------------------------------------------------------------
before_rows = len(df_clean)

df_gf_test = df_clean.merge(
    gf_bio_lookup,
    on="COSMIC_ID",
    how="left"
)

after_rows = len(df_gf_test)

print("\nSafe merge test:")
print("Rows before merge:", before_rows)
print("Rows after merge :", after_rows)

# ------------------------------------------------------------
# 6) Coverage and missingness summary
# ------------------------------------------------------------
added_cols = [c for c in gf_bio_lookup.columns if c != "COSMIC_ID"]

non_missing_counts = df_gf_test[added_cols].notna().sum().sort_values(ascending=False)
missing_pct = (df_gf_test[added_cols].isna().mean() * 100).sort_values()

print("\nNon-missing counts for selected biological/genomic columns:")
print(non_missing_counts)

print("\nMissing % for selected biological/genomic columns:")
print(missing_pct)

# ------------------------------------------------------------
# 7) Data type and value preview
# ------------------------------------------------------------
print("\nData types of selected biological/genomic columns:")
print(df_gf_test[added_cols].dtypes)

for col in added_cols:
    print(f"\nSample values in {col}:")
    print(df_gf_test[col].dropna().astype(str).value_counts().head(10))

Original genomic_features shape: (2266, 96)
Original unique COSMIC_IDs: 1122
Rows with missing COSMIC_ID: 1144

After dropping missing COSMIC_ID rows:
Shape: (1122, 96)
Unique COSMIC_IDs: 1122

Selected biological/genomic columns:
['COSMIC_ID', 'ploidy_snp6', 'ploidy_wes', 'ploidy_wgs', 'mutational_burden', 'mlh1_expression_by_ihc', 'mlh1_promoter_methylation_status', 'msh2_expression_by_ihc', 'pms2_expression_by_ihc', 'msh6_expression_by_ihc', 'braf_mutation_identified', 'braf_expression_by_ihc', 'pik3ca_mutation_identified', 'pten_expression_by_ihc', 'pten_mutation_identified', 'kras_mutation_identified', 'mismatch_repair_status']
Number of selected columns (including COSMIC_ID): 17

Number of duplicated valid COSMIC_IDs in selected genomic data: 0
No duplicated valid COSMIC_IDs detected in selected genomic data.

Collapsed biological/genomic lookup shape: (1122, 17)
Unique COSMIC_IDs in collapsed lookup: 1122

Safe merge test:
Rows before merge: 242035
Rows after merge : 242035

Non

In [30]:
# Final genomic feature selection step

# Start from the selected genomic lookup from the previous step
genomic_cols = [c for c in gf_bio_lookup.columns if c != "COSMIC_ID"]

# Calculate coverage in the merged test table
coverage_pct = (df_gf_test[genomic_cols].notna().mean() * 100).sort_values(ascending=False)

print("Coverage % for selected genomic columns:")
print(coverage_pct)

# Choose a minimum coverage threshold
min_coverage_pct = 20

genomic_cols_keep = coverage_pct[coverage_pct >= min_coverage_pct].index.tolist()
genomic_cols_drop = coverage_pct[coverage_pct < min_coverage_pct].index.tolist()

print(f"\nColumns kept (>= {min_coverage_pct}% non-missing):")
print(genomic_cols_keep)

print(f"\nColumns dropped (< {min_coverage_pct}% non-missing):")
print(genomic_cols_drop)

# Build final genomic feature lookup with only kept columns
gf_bio_final = gf_bio_lookup[["COSMIC_ID"] + genomic_cols_keep].copy()

print("\nFinal genomic feature lookup shape:", gf_bio_final.shape)

# Merge final selected genomic features into df_clean
before_rows = len(df_clean)

df_clean = df_clean.merge(
    gf_bio_final,
    on="COSMIC_ID",
    how="left"
)

after_rows = len(df_clean)

print("\nFinal merge check:")
print("Rows before merge:", before_rows)
print("Rows after merge :", after_rows)

print("\nFinal genomic columns added to df_clean:")
print(genomic_cols_keep)

print("\nMissing counts in added genomic columns:")
print(df_clean[genomic_cols_keep].isna().sum())

Coverage % for selected genomic columns:
mutational_burden                   99.769868
ploidy_snp6                         99.219948
ploidy_wes                          99.208379
ploidy_wgs                           0.000000
mlh1_expression_by_ihc               0.000000
mlh1_promoter_methylation_status     0.000000
msh2_expression_by_ihc               0.000000
pms2_expression_by_ihc               0.000000
msh6_expression_by_ihc               0.000000
braf_mutation_identified             0.000000
braf_expression_by_ihc               0.000000
pik3ca_mutation_identified           0.000000
pten_expression_by_ihc               0.000000
pten_mutation_identified             0.000000
kras_mutation_identified             0.000000
mismatch_repair_status               0.000000
dtype: float64

Columns kept (>= 20% non-missing):
['mutational_burden', 'ploidy_snp6', 'ploidy_wes']

Columns dropped (< 20% non-missing):
['ploidy_wgs', 'mlh1_expression_by_ihc', 'mlh1_promoter_methylation_status', 'msh2_

Final helper-column cleanup
- This step removes the temporary columns that were only needed during preprocessing, such as helper merge columns, fill flags, and intermediate remap columns. We do this now so the dataset only keeps meaningful final variables, and so temporary preprocessing artifacts do not accidentally get treated as model features later.The code builds a list of known helper and stage-specific columns, keeps only the ones that actually exist in df_clean, drops them, and then prints the remaining missingness for the main cleaned columns. That gives you a cleaner dataset and a final missingness check before defining the modeling feature pool.

In [31]:
# Final helper-column cleanup

helper_cols_to_drop = [
    # helper merge columns
    "msi_status",
    "growth_properties",
    "cancer_type",
    "GDSC_Tissue_descriptor_1_df2",
    "GDSC_Tissue_descriptor_2_df2",
    "TCGA_DESC_df2",
    "tcga_from_desc2_unique",
    "stage3_resolved_tcga",

    # fill/remap flags
    "descriptor1_was_imputed",
    "descriptor2_was_imputed",
    "tcga_missing_before_stage1",
    "tcga_filled_from_cancer_type",
    "tcga_filled_manually",
    "tcga_filled_from_desc2",
    "tcga_other_to_prad",
    "tcga_stage2_exception_to_unclassified",
    "tcga_resolved_from_stage3",
]

# Keep only columns that actually exist
helper_cols_to_drop = [col for col in helper_cols_to_drop if col in df_clean.columns]

print("Helper/temp columns to drop:")
print(helper_cols_to_drop)

df_clean = df_clean.drop(columns=helper_cols_to_drop)

print("\ndf_clean shape after helper cleanup:", df_clean.shape)

# Final missingness check on key cleaned columns
key_check_cols = [
    "LN_IC50",
    "DRUG_NAME",
    "TARGET",
    "TCGA_DESC",
    "Microsatellite instability Status (MSI)",
    "Growth Properties",
    "GDSC Tissue descriptor 1",
    "GDSC Tissue descriptor 2",
    "mutational_burden",
    "ploidy_snp6",
    "ploidy_wes",
]

key_check_cols = [col for col in key_check_cols if col in df_clean.columns]

print("\nMissing values after helper cleanup:")
print(df_clean[key_check_cols].isna().sum())

Helper/temp columns to drop:
['cancer_type', 'GDSC_Tissue_descriptor_1_df2', 'GDSC_Tissue_descriptor_2_df2', 'tcga_from_desc2_unique', 'stage3_resolved_tcga', 'descriptor1_was_imputed', 'descriptor2_was_imputed', 'tcga_missing_before_stage1', 'tcga_filled_from_cancer_type', 'tcga_filled_manually', 'tcga_filled_from_desc2', 'tcga_other_to_prad', 'tcga_resolved_from_stage3']

df_clean shape after helper cleanup: (242035, 23)

Missing values after helper cleanup:
LN_IC50                                        0
DRUG_NAME                                      0
TARGET                                     27872
TCGA_DESC                                      0
Microsatellite instability Status (MSI)      630
Growth Properties                              0
GDSC Tissue descriptor 1                       0
GDSC Tissue descriptor 2                       0
mutational_burden                            557
ploidy_snp6                                 1888
ploidy_wes                                  1

Choose final feature set - 
- This step decides which columns are part of the actual modeling dataset and which ones are just identifiers, labels, or non-feature fields. We do this before train/test split so you and your friend have a clean candidate feature pool, while still leaving the deeper feature-selection work for later.


In [32]:
# Choose final feature set (basic modeling pool, not statistical feature selection yet)

target_col = "LN_IC50"

# Keep for reference, but not as predictors
id_cols = [
    "COSMIC_ID",
    "CELL_LINE_NAME",
    "DRUG_ID",
]

id_cols = [col for col in id_cols if col in df_clean.columns]

# Candidate predictors to carry forward
candidate_feature_cols = [
    # drug-related
    "DRUG_NAME",
    "TARGET",
    "TARGET_PATHWAY",

    # cancer/tissue-related
    "TCGA_DESC",
    "Microsatellite instability Status (MSI)",
    "Growth Properties",
    "GDSC Tissue descriptor 1",
    "GDSC Tissue descriptor 2",

    # selected genomic features
    "mutational_burden",
    "ploidy_snp6",
    "ploidy_wes",
]

candidate_feature_cols = [col for col in candidate_feature_cols if col in df_clean.columns]

print("Target column:")
print(target_col)

print("\nID/reference columns kept out of predictors:")
print(id_cols)

print("\nCandidate feature columns:")
print(candidate_feature_cols)

# Build modeling pieces
X_candidate = df_clean[candidate_feature_cols].copy()
y = df_clean[target_col].copy()

# Optional combined dataset for export/review
modeling_df = df_clean[id_cols + candidate_feature_cols + [target_col]].copy()

print("\nX_candidate shape:", X_candidate.shape)
print("y shape:", y.shape)
print("modeling_df shape:", modeling_df.shape)

print("\nMissing values in candidate features:")
print(X_candidate.isna().sum().sort_values(ascending=False))

print("\nData types in candidate features:")
print(X_candidate.dtypes)

Target column:
LN_IC50

ID/reference columns kept out of predictors:
['COSMIC_ID', 'CELL_LINE_NAME', 'DRUG_ID']

Candidate feature columns:
['DRUG_NAME', 'TARGET', 'TARGET_PATHWAY', 'TCGA_DESC', 'Microsatellite instability Status (MSI)', 'Growth Properties', 'GDSC Tissue descriptor 1', 'GDSC Tissue descriptor 2', 'mutational_burden', 'ploidy_snp6', 'ploidy_wes']

X_candidate shape: (242035, 11)
y shape: (242035,)
modeling_df shape: (242035, 15)

Missing values in candidate features:
TARGET                                     27872
ploidy_wes                                  1916
ploidy_snp6                                 1888
Microsatellite instability Status (MSI)      630
mutational_burden                            557
TARGET_PATHWAY                                 0
DRUG_NAME                                      0
GDSC Tissue descriptor 1                       0
Growth Properties                              0
TCGA_DESC                                      0
GDSC Tissue descriptor

Train/Split -
- This step divides the cleaned modeling dataset into a training set and a test set. We do this now so any later steps like encoding, scaling, or feature selection are learned only from the training data, which helps prevent data leakage. The code uses train_test_split to separate X_candidate and y into training and test sets, then prints the shapes and checks that the target distribution is still similar between the two splits. I am using a standard 80/20 split and a fixed random_state so your group can reproduce the same split later.


In [33]:
from sklearn.model_selection import train_test_split

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_candidate,
    y,
    test_size=0.20,
    random_state=42
)

print("Train/test split complete.")
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

# Quick target distribution check
print("\nLN_IC50 summary:")
print("y_train mean:", y_train.mean())
print("y_test mean :", y_test.mean())
print("y_train std :", y_train.std())
print("y_test std  :", y_test.std())

Train/test split complete.
X_train shape: (193628, 11)
X_test shape : (48407, 11)
y_train shape: (193628,)
y_test shape : (48407,)

LN_IC50 summary:
y_train mean: 2.8161668489371374
y_test mean : 2.820886711963146
y_train std : 2.762214428002262
y_test std  : 2.7621209974790277
